In [13]:
from typing import Annotated

from langchain_core.messages import AnyMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [14]:
load_dotenv()

True

In [15]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [16]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [17]:
def chat_node(state:ChatState):
    decision=interrupt({
        "type":"approval",
        "reason":"Model is about a answer a question",
        "question":state["messages"][-1].content,
        "instruction":"Approve this question ? yes/no"
    })

    if decision["approved"]=='no':
        return {"messages":[AIMessage(content="Not approved")]}
    else:
        response=llm.invoke(state['messages'])
        return {"messages":[response]}

In [18]:
graph=StateGraph(ChatState)

graph.add_node('chat',chat_node)
graph.add_edge(START,'chat')
graph.add_edge('chat',END)

checkpointer=MemorySaver()
app=graph.compile(checkpointer=checkpointer)


In [26]:
config={"configurable":{
    "thread_id":"thread_1"
}}

inital_input={
    "messages":[
        ("user","What is the capital of France?")
    ]
}

In [27]:
result=app.invoke(inital_input,config=config)
result

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='16b1b7df-023d-4bad-a809-3d575dfbc69e'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e9c9-9869-7cb0-b502-1598a5f0c634-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 23, 'total_tokens': 31, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 15}}),
  HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='02a6facc-495e-4920-a2e3-dca2e699352c')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about a answer a question', 'question': 'What is the capital of France?', 'instruction': 'Approve this question ? yes/no'}, id='6be0bb45a5b3166fdbca346

In [28]:
message = result['__interrupt__'][0].value

In [29]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [30]:
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [31]:
print(final_result["messages"][-1].content)

Not approved
